In [ ]:
!mkdir -p week-2
%cd week-2
!pwd

/content/week-2
/content/week-2


In [ ]:
!apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Selecting previously unselected package zstd.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
!ollama --version

In [ ]:
import subprocess
import time
import requests

ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

for i in range(30):
    try:
        response = requests.get("http://localhost:11434")
        if response.ok:
            print("SUCCESS:", response.text)
            break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    print("Server failed to start")

SUCCESS: Ollama is running


In [ ]:
!ollama pull llama3.2

In [ ]:
!ollama list

NAME               ID              SIZE      MODIFIED           
llama3.2:latest    a80c4f17acd5    2.0 GB    About a minute ago    


In [ ]:
!curl -s http://localhost:11434/api/generate -d '{"model":"llama3.2","prompt":"Say hello in one short sentence.","stream":false}'

{"model":"llama3.2","created_at":"2026-11-16T08:24:38.885450292Z","response":"Hello!","done":true,"done_reason":"stop","context":[128006,9125,128007,271,38766,1303,33025,2696,25,6790,220,2366,18,271,128009,128006,882,128007,271,46864,24748,304,832,2875,11914,13,128009,128006,78191,128007,271,9906,0],"total_duration":114188327928,"load_duration":60780516208,"prompt_eval_count":32,"prompt_eval_cached_count":0,"prompt_eval_duration":53364341000,"eval_count":3,"eval_duration":40514000}

In [ ]:
%cd /content/week-2

with open(".env", "w") as f:
    f.write(f'GEMINI_API_KEY="{key}"\n')

print(".env created successfully!")

/content/week-2
.env created successfully!


In [ ]:
%%writefile requirements.txt
python-dotenv
pytest
google-genai
requests

Writing requirements.txt


In [ ]:
!pip install -q -r requirements.txt

In [ ]:
from google import genai

print("google-genai imported OK")

google-genai imported OK


In [ ]:
%%writefile llm_client.py

from abc import ABC, abstractmethod
from google import genai
import requests


class LLMProvider(ABC):

    @abstractmethod
    def generate(self, prompt):
        pass


class GeminiProvider(LLMProvider):

    def __init__(self, api_key, model="gemini-3.8-flash"):
        self.client = genai.Client(api_key=api_key)
        self.model = model

    def generate(self, prompt):
        response = self.client.models.generate_content(
            model=self.model,
            contents=prompt
        )
        return response.text


class OllamaProvider(LLMProvider):

    def __init__(self, model="llama3.2"):
        self.model = model
        self.url = "http://localhost:11434/api/generate"

    def generate(self, prompt):
        response = requests.post(
            self.url,
            json={
                "model": self.model,
                "prompt": prompt,
                "stream": False
            },
            timeout=120
        )

        response.raise_for_status()
        return response.json()["response"]


class LLMClient:

    def __init__(self, provider):
        self.provider = provider

    def generate(self, prompt):
        return self.provider.generate(prompt)

Overwriting llm_client.py


In [ ]:

%%writefile main.py

import os
from dotenv import load_dotenv

from llm_client import (
    GeminiProvider,
    OllamaProvider,
    LLMClient
)

load_dotenv(override=True)

provider_name = input(
    "Choose provider (gemini/ollama): "
).strip().lower()

if provider_name == "gemini":

    api_key = os.getenv("GEMINI_API_KEY")

    if not api_key:
        raise ValueError("Gemini API key not found in .env")

    provider = GeminiProvider(api_key)

elif provider_name == "ollama":

    provider = OllamaProvider("llama3.2")

else:
    raise ValueError("Invalid provider selected")

client = LLMClient(provider)

prompt = input("Enter your question: ")

answer = client.generate(prompt)

print("\nLLM Response:")
print(answer)


Writing main.py


In [ ]:

%%writefile test_client.py

import unittest
from llm_client import LLMClient


class FakeProvider:

    def generate(self, prompt):
        return "AI stands for Artificial Intelligence."


class TestLLMClient(unittest.TestCase):

    def test_generate(self):

        provider = FakeProvider()
        client = LLMClient(provider)

        result = client.generate("Explain AI")

        self.assertEqual(
            result,
            "AI stands for Artificial Intelligence."
        )


if __name__ == "__main__":
    unittest.main()


Writing test_client.py


In [ ]:
!python -m pytest test_client.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/week-2
plugins: anyio-4.16.0, platformdirs-4.14.3, langsmith-0.14.17, typeguard-4.6.0
collected 1 item                                                               

test_client.py::TestLLMClient::test_generate PASSED                      [100%]

============================== 1 passed in 1.30s ===============================


In [ ]:
%run main.py

Choose provider (gemini/ollama): ollama
Enter your question: Explain artificial intelligence in simple words.

LLM Response:
Artificial intelligence (AI) is a way to make machines think and learn like humans. It's like creating a computer program that can solve problems, make decisions, and even learn from its mistakes.

Think of it like this: humans have brains that can understand language, recognize patterns, and adapt to new situations. AI is a computer program that tries to do the same things, but with code and data instead of brains.

There are three main types of AI:

1. **Narrow AI**: This type of AI is good at one specific task, like recognizing faces in pictures or translating languages.
2. **General AI**: This type of AI is good at many different tasks, like playing chess or answering questions.
3. **Superintelligent AI**: This type of AI is much, much smarter than humans and can solve complex problems that we can't even imagine.

AI works by using algorithms (like recipes fo

In [ ]:
%run main.py

Choose provider (gemini/ollama): gemini
Enter your question: Explain artificial intelligence in simple words.

LLM Response:
**Artificial Intelligence (AI)** is simply when we make computers and machines smart enough to do things that usually require human brainpower.

Think of it like this:

### 1. Normal Computers vs. AI
* **A normal computer** is like a faithful cook who only follows a recipe. If you don't write down every single step, it gets confused and stops working.
* **AI** is like a cook who tastes thousands of different meals, learns what tastes good, and can start inventing its own recipes without being told every single step.

### 2. How does it learn?
AI learns by looking at huge amounts of examples—just like a child does. 

> *For example:* If you want an AI to recognize a **cat**, you don't explain to it what a cat is. Instead, you show it millions of photos of cats. Eventually, the computer notices patterns (pointy ears, whiskers, fur) and can say, *"Aha! That's a cat!